In [ ]:
from genpm.utils.consts import SHARED_DIR_PATH, SPARK_CONFIGS
from genpm.utils.spark_session import SparkDataManager

from pyspark.sql import functions as f

sdm = SparkDataManager(additional_conf=SPARK_CONFIGS["STANDARD_FIFTH"])
DF_PATH = SHARED_DIR_PATH / "preprocessed_dataset" / "final_pmcm" / "pm_df_wide_indexed_winds"

In [ ]:
pm_df = sdm.read_parquet(DF_PATH)

In [ ]:
pm_df.printSchema()

In [ ]:
pm_df.filter(f.col("distname") == "bts_14/cell_4").orderBy("start_time").show(49)

In [ ]:
pm_df.select([f.sum(f.col(c).isNull().cast("int").alias(c)) for c in pm_df.columns]).show()

In [ ]:

WINDOW_WIDTH = 168

_META = {"distname", "bts_id", "window_anchor", "start_time"}
CONFIG_COLS = [c for c in pm_df.columns if c.startswith("[CELL]")]
# Identity: what makes two windows belong to different series
GROUP_IDENTITY_COLS = ["distname", "bts_id", *CONFIG_COLS]
KPI_COLS = [c for c in pm_df.columns if c not in _META and c not in CONFIG_COLS]


def materialize_windows(df, identity_cols, kpi_cols, window_width=WINDOW_WIDTH, out_path=None):
    """
    For each (identity_cols, window_anchor) anchor, collect all rows whose
    start_time falls in [window_anchor, window_anchor + window_width * 3600 s).

    Output schema:
        *identity_cols, window_anchor, hour_idx, kpi_1, kpi_2, ...
    One row per hour per window, ordered by hour_idx (0 = anchor hour).
    """
    # Step 1: one row per (identity, window_anchor) — the anchor definition
    anchors = (
        df.filter(f.col("window_anchor").isNotNull())
        .select(*identity_cols, "window_anchor")
        .distinct()
        .alias("anc")
    )

    data = df.alias("data")

    # Step 2: range join — match identity columns + time window
    identity_join = [f.col(f"data.{c}") == f.col(f"anc.{c}") for c in identity_cols]
    anc_epoch  = f.col("anc.window_anchor").cast("long")
    data_epoch = f.col("data.start_time").cast("long")
    range_join = [
        data_epoch >= anc_epoch,
        data_epoch <  anc_epoch + window_width * 3600,
    ]

    result = (
        data.join(anchors, identity_join + range_join, "inner")
        .select(
            *[f.col(f"anc.{c}").alias(c) for c in identity_cols],
            f.col("anc.window_anchor").alias("window_anchor"),
            ((data_epoch - anc_epoch) / 3600).cast("int").alias("hour_idx"),
            *[f.col(f"data.{c}").alias(c) for c in kpi_cols],
        )
        .orderBy(*identity_cols, "window_anchor", "hour_idx")
    )

    if out_path is not None:
        result.write.mode("overwrite").parquet(str(out_path))
        print(f"Saved -> {out_path}")

    return result


OUT_PATH = SHARED_DIR_PATH / "preprocessed_dataset" / "final_pmcm" / "pm_df_wide_materialized_windows"

materialized_df = materialize_windows(pm_df, GROUP_IDENTITY_COLS, KPI_COLS, out_path=OUT_PATH).cache()
materialized_df.printSchema()


In [ ]:
materialized_df = materialized_df.cache()
materialized_df.count()

In [ ]:
materialized_df.show(400)